In [1]:
# 環境架設
from selenium import webdriver
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.chrome.service import Service as ChromeService

import module.xpon_module as xpon
import module.selenium_module as mod
import module.xpon_olt_status_module as olts
import module.xpon_olt_config_module as oltc
import module.xpon_batch_ont_config_module as bat
import module.xpon_ont_config_module as ontc
import module.xpon_profiles_module as pro

# copy from GLC project-------------
import module.acl_module as acl  
import module.glc_test_module as glc
# ----------------------------------

from selenium.webdriver.support.ui import Select
import time
import datetime

import module.ems_acl_module as e_acl
import module.ems_profile_module as e_pro
import module.ems_test_module as ems

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from module.customdriver import CustomDriver
import module.globals
import module.get_card
import json
import os
import argparse
# create "Report" folder if not exist
if not os.path.exists("Report"):
    os.makedirs("Report")

# SET time_sleep_before_find_element
import os
if not os.path.isfile( os.path.join('ENV', 'time_sleep_before_find_element.txt') ):
    with open(os.path.join('ENV', 'time_sleep_before_find_element.txt'), 'w') as file:
        file.write("time_sleep_before_find_element = '0.3'")
_0_set_up_file = open("ENV\\time_sleep_before_find_element.txt",'r')
_0_lines = _0_set_up_file.readlines()
_0_set_up_file.close()
time_sleep_before_find_element = float(_0_lines[0].split("'")[1])    



In [2]:
env_file = "ENV/ENV_WEB_202.json"

In [3]:
with open(env_file, 'r', encoding='utf-8') as f:
    d = json.load(f)

url = d["ZYXEL_DUT"]["NODE1"]["url"]
url_ip = url.split("/")[2]
username = d['ZYXEL_DUT']['NODE1']["login_username"]
password = d['ZYXEL_DUT']['NODE1']["login_password"]
default_username = d['ZYXEL_DUT']['NODE1']["default_username"]
default_password = d['ZYXEL_DUT']['NODE1']["default_password"]
device_name = d['ZYXEL_DUT']['NODE1']["name"]
map_name = d['ZYXEL_DUT']['NODE1']["map_name"]
set_community = d['ZYXEL_DUT']['NODE1']["set_community"]
get_community = d['ZYXEL_DUT']['NODE1']["get_community"]
loop = d["LOOP"]["count"]

# EMS
ems_url = d["EMS"]["url"]
ems_username = d["EMS"]["login_username"]
ems_password = d["EMS"]["login_password"]
ems.set_ems_credentials(ems_url, ems_password)  # 設定本 thread 的 EMS 連線資訊

# Card
msc_slot = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["MSC"]["slot_id"]
msc_port = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["MSC"]["port_id"]
glc_slot = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["GLC1440X"]["slot_id"]
glc_port = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["GLC1440X"]["port_id"]

# GPON (未修改)
slot_1_aid = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["OLC3416-42A"]["slot_id"]
pon_1_aid = d["ZYXEL_DUT"]["NODE1"]["CARDINFO"]["OLC3416-42A"]["port_id"]

# XPON
slot_x_1_aid = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["slot_id"]
pon_x_1_aid = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["port_id"]
ont_x_1_1 = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["ont_id"]
ont_x_1_1_aid = f'{slot_x_1_aid}-{pon_x_1_aid}-{ont_x_1_1}'

ont_x_1_1_sn_12 = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["sn_12"]
ont_x_1_1_sn = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["sn_16"]
ont_x_1_1_pw = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["pw"]
ont_x_1_1_ont_USER = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["ont_user"]
ont_x_1_1_ont_PW = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["ont_pw"]

service_mode_x_1 = d["ZYXEL_DUT"]["NODE1"]["ONTINFO"]["XPON"]["ONT1"]["service_mode"]
service_mode_x_2 = service_mode_x_1
inband_vlan = d["ZYXEL_DUT"]["NODE1"]["inband_vlan"]

snmp_get = ''
snmp_set = ''
snmp_version = ''
test_retry_num = 2

# === Test Start ===
# 每個 Thread 自己建 driver，避免共用衝突
driver = CustomDriver(time_sleep=time_sleep_before_find_element)

start_time = time.time()

# --- FW Web CardType Case ---
driver.get(url)
mod.Login(driver, username, password)
fw_title = driver.title

if "Neox".lower() in driver.title.lower():
    if_Neox = True
else:
    if_Neox = False
ems.parameter_add("if_Neox", if_Neox)

mod.Switch_Default(driver)
mod.Page_Switch(driver, "Home")
time.sleep(5)
mod.Switch_Frame(driver)
msc_card_type = driver.find_elements(
    'class name', "td-table-content.SysinfoCardType.left-wrapper"
)[int(msc_slot) - 1].get_attribute("textContent")
module.globals.msc_card_type = msc_card_type  # test
msc_version = driver.find_elements(
    'class name', "td-table-content.Sysinfofwversion.left-wrapper"
)[int(msc_slot) - 1].get_attribute("textContent")

# ems.Maintenance_Config_Reset(driver)  # 如需

ems.EMS_Set_Prepare(driver, snmp_get, snmp_set, snmp_version, username, password)

list1 = module.get_card.get_card_case(driver,glc_port,pon_x_1_aid,pon_1_aid,msc_port,file_name='')

ems.FW_Logout(driver)

# --- EMS Web ---
driver.get(ems_url)
try:
    driver.maximize_window()
except:
    pass

time.sleep(5)
mod.Login_EMS(driver, username, ems_password)
time.sleep(5)
try:
    driver.find_element('id', 'helpMenu').click()
except:
    mod.Login_EMS(driver, username, ems_password)
    time.sleep(5)
    driver.find_element('id', 'helpMenu').click()

driver.find_element('id', 'about_ems_button').click()
time.sleep(3)
emsversion = driver.find_element('id', 'emsversion').text.split("\n")[0]
time.sleep(5)
driver.find_element('xpath', '//*[@type="button" and @class="button button_green"]').click()

# --- Set File Name（每個執行緒各寫自己的檔案，避免互卡）---
date1 = str(datetime.datetime.now()).split(":")[0]
date2 = str(datetime.datetime.now()).split(":")[1]
file_name = f"Report/Web_EMS_Config_Card_TMP_{fw_title}_{emsversion}_{date1}{date2}_{url_ip[-3:]}.log"
ems.parameter_add("file_name", file_name)
ems.get_file_name(file_name)

# --- Info in File ---
print("EMS Web Auto Report", file=open(file_name, '+a', encoding='utf-8'))
print(datetime.datetime.now(), file=open(file_name, '+a', encoding='utf-8'))
print(f"EMS Web IP:{ems_url}  \nEMS Version:{emsversion} ", file=open(file_name, '+a', encoding='utf-8'))
print(f"FW Web IP:{url}FW MSC:{msc_slot}-{msc_port}/{msc_card_type}/{msc_version}", file=open(file_name, '+a', encoding='utf-8'))
print(f"if_Neox: {ems.parameter_get('if_Neox')}", file=open(file_name, '+a', encoding='utf-8'))
print(f"{list1}", file=open(file_name, '+a', encoding='utf-8'))

MSC_slot = list1.pop()
MSC_card = list1.pop()


module.get_card.print_case(list1, file_name)

print(f"env file:{env_file}", file=open(file_name, '+a', encoding='utf-8'))
print(f"  (find_element(), find_elements() will wait {time_sleep_before_find_element} sec )\n", file=open(file_name, '+a', encoding='utf-8'))

print(f"EMS Test\nTest Start", file=open(file_name, '+a', encoding='utf-8'))


dict1:{'NXA340': '1', 'NXC400': '4', 'NXP316': '8'}
dict_keys(['NXA340', 'NXC400', 'NXP316'])
slot:1 NXA340, in case (1) GLC1440X, GLC1540X
slot:4 NXC400, in case (4) MSC
slot:8 NXP316, in case (2) OLC3416B, OLC3708, OLC3816 (XPON)
list1:['1,1,39,NXA340', '4,4,24,NXC400', '2,8,16,NXP316']
list1:['1,1,39,NXA340', '4,4,24,NXC400', '2,8,16,NXP316', 'NXC400', '4']
             [Statistics - Online Users] session:*http-1


In [4]:
print("======================================================================================", file=open(file_name, '+a', encoding='utf-8'))
print("Card Test", file=open(file_name, '+a', encoding='utf-8'))
print("======================================================================================", file=open(file_name, '+a', encoding='utf-8'))

for i in range(len(list1)):
    list_element = list1[i]
    slot = list_element.split(",")[1]
    port = list_element.split(",")[2]
    cardtype = list_element.split(",")[3]


    print(f"\n>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> [ {cardtype} ]",file = open(file_name, '+a'))
    print(
        f"             Now in case:{list_element[:1]},  Slot-Port:{slot}-{port}, {cardtype}   ({i+1}/{len(list1)})",
        file=open(file_name, '+a', encoding='utf-8')
    )
    print(
        f"             Now in case:{list_element[:1]},  Slot-Port:{slot}-{port}, {cardtype}   ({i+1}/{len(list1)})"
        
    )
    print("======================================================================================", file=open(file_name, '+a', encoding='utf-8'))
    
    # # ACL

             Now in case:1,  Slot-Port:1-39, NXA340   (1/3)
             Now in case:4,  Slot-Port:4-24, NXC400   (2/3)
             Now in case:2,  Slot-Port:8-16, NXP316   (3/3)


In [ ]:
# list1 = ['1,1,39,NXA340', '4,4,24,NXC400', '2,8,16,NXP316']

['1,1,39,NXA340', '4,4,24,NXC400', '2,8,16,NXP316']

In [7]:
list_element = list1[0]
slot = list_element.split(",")[1]
port = list_element.split(",")[2]
cardtype = list_element.split(",")[3]
print(f"             ({cardtype} {slot}-{port})")

             (NXA340 1-39)


In [ ]:
# ems.EMS_ACL_Rule_Profile(driver,url,list_element,username,password, ems_url, ems_password,file_name)


In [ ]:
from module.ems_test_module import *



if_NeoX = False
if_EMS_error = False
if_FW_error = False
if_pro_new_error = False
if_pro_mode_error = False
if_pro_modify_error = False
if_pro_delete_error = False
if_ru_profile_error = False  # Mode:Profile
if_ru_priority_error = False # Mode:Port
if_ru_delete_error = False
# --------------------- EMS web start ---------------------
print(f"             [ACL - ACL Profile] EMS (Set ACL Profile and Check)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(3)
print(f"             [ACL - ACL Profile] EMS set Mode:Port",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Name:1234567890123456789012345678001",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Source MAC/OUI:10:10:10:10:10:01",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Drop:True",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Profile-Priority:1",file = open(file_name, '+a'))
e_acl.ACL_Profile_Mode(driver,"Port")
e_acl.ACL_Profile_Mode_Apply(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Port  Apply msg:{msg}  Apply Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_mode_error = True
else:
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Port  Apply msg:{msg}  Apply Pass",file = open(file_name, '+a'))
e_acl.ACL_Profile_New(driver)
# e_acl.ACL_Profile_New_Name(driver,"1234567890123456789012345678001")
e_acl.ACL_Profile_New_Name(driver,"12345678901234567890_priority_1")
e_acl.ACL_Profile_New_Source_MAC_OUI(driver,"10:10:10:10:10:01")
e_acl.ACL_Profile_New_Drop(driver)
e_acl.ACL_Profile_New_Profile_Priority(driver,"1")
e_mod.OK(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS Apply Profile 12345678901234567890_priority_1 msg:{msg}  Apply Fail Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_new_error = True    
else:
    print(f"             [ACL - ACL Profile] EMS Apply Profile 12345678901234567890_priority_1 msg:{msg}  Apply Pass Test Pass",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Profile-Priority:False",file = open(file_name, '+a'))
e_acl.ACL_Profile_New(driver)
e_acl.ACL_Profile_New_Name(driver,"1234567890123456789012345678001")
e_acl.ACL_Profile_New_Source_MAC_OUI(driver,"10:10:10:10:10:01")
e_acl.ACL_Profile_New_Drop(driver)
e_acl.ACL_Profile_New_Profile_Priority(driver,False)
e_mod.OK(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789012345678001 msg:{msg}  Apply Fail Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_new_error = True   
else:
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789012345678001 msg:{msg}  Apply Pass Test Pass",file = open(file_name, '+a'))



# Table Page Switch Test
if_page_error = False
print(f"\n             [ACL - ACL Profile] EMS (table page switch test)",file = open(file_name, '+a'))
try:
    for i in range(2,12):
        time.sleep(3)
        n = str(int("1234567890123456789012345678000") + i)
        e_acl.ACL_Profile_New(driver)
        e_acl.ACL_Profile_New_Name(driver,n)
        e_mod.OK(driver)
        msg = alert_accept_text(driver)
        if(if_msg_not_success(msg)):        
            print(f"             [ACL - ACL Profile] EMS Apply Profile {n} msg:{msg}  Apply Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_page_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Apply Profile {n} msg:{msg}  Apply Pass",file = open(file_name, '+a'))
except:
    print(f"             [ACL - ACL Profile] EMS New Profile (2~11) Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_page_error = True
e_mod.NextPage_Click(driver)
table_row = driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()
if table_row == 0:
    print(f"             [ACL - ACL Profile] EMS Table Row Num {table_row}=0 After NextPage Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_page_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Table Row Num {table_row}!=0 After NextPage Test Pass",file = open(file_name, '+a'))
# if if_page_error:
#     print(f"  [EMS1-6149][ACL - ACL Profile - Table Page Switch] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6149][ACL - ACL Profile - Table Page Switch] Result Pass",file = open(file_name, '+a'))

# Check
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(3)
if(Select(driver.find_element(By.ID,"acl_AclProfile_mode")).first_selected_option.text != "Port"):
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : {Select(driver.find_element(By.ID,'acl_AclProfile_mode')).first_selected_option.text}!=Port Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_mode_error = True
else:
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Port Test Pass",file = open(file_name, '+a'))
time.sleep(1)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678001":
        if_find = True
        e_acl.ACL_Profile_Select(driver,i+1)
        e_acl.ACL_Profile_Modify(driver)
        time.sleep(1)
        if(driver.find_element(By.ID,"acl_aclprofile_name_text").get_attribute("value") != "1234567890123456789012345678001"):
            print(f"             [ACL - ACL Profile] EMS Name {driver.find_element(By.ID,'acl_aclprofile_name_text').get_attribute('value')}!=1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Name 1234567890123456789012345678001 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.ID,"acl_aclprofile_sourmac_text").get_attribute("value") != "10:10:10:10:10:01"):
            print(f"             [ACL - ACL Profile] EMS Source MAC/OUI {driver.find_element(By.ID,'acl_aclprofile_sourmac_text').get_attribute('value')}!=10:10:10:10:10:01 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Source MAC/OUI 10:10:10:10:10:01 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.ID,"acl_aclprofile_drop_checkbox").is_selected() == False):
            print(f"             [ACL - ACL Profile] EMS Drop {driver.find_element(By.ID,'acl_aclprofile_drop_checkbox').is_selected()}!=True Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Drop True Test Pass",file = open(file_name, '+a'))                
        # if(driver.find_element(By.ID,"acl_aclprofile_pp_text").get_attribute("value") != '1'):
        #     print(f"             [ACL - ACL Profile] EMS Profile-Priority {driver.find_element(By.ID,'acl_aclprofile_pp_text').get_attribute('value')}!=1 Test Fail",file = open(file_name, '+a'))
        #     if_EMS_error = True
        #     if_pro_new_error = True
        # else:
        #     print(f"             [ACL - ACL Profile] EMS Profile-Priority 1 Test Pass",file = open(file_name, '+a'))    
        break
if not if_find:
    print(f"             [ACL - ACL Profile] EMS Not Found Profile Name 1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))    
    if_EMS_error = True
# --------------------- EMS web end ---------------------
# Check ACL Profile
# --------------------- FW web start ---------------------
# Profile > ACL 
driver.get(url) 
time.sleep(3)
if "NeoX" in driver.title:
    if_NeoX = True    
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"Profile")
# mod.SubPage_Switch(driver,"ACL")
mod.Switch_Frame(driver)
time.sleep(2)    
if(Select(driver.find_element(By.NAME,"rpaclprof_aclProfileMode")).first_selected_option.text != "Port"):
    print(f"             [Profile - ACL] FW ACL Profile Mode : {Select(driver.find_element(By.NAME,'rpaclprof_aclProfileMode')).first_selected_option.text}!=Port Test Fail",file = open(file_name, '+a'))
    if_FW_error = True
    if_pro_mode_error = True
else:
    print(f"             [Profile - ACL] FW ACL Profile Mode : Port Test Pass",file = open(file_name, '+a'))
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName").__len__()):
    if driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName")[i].text == '1234567890123456789012345678001':
        if_find = True
        driver.find_elements(By.NAME,"rpaclprof_rdoSlt")[i].click()
        if(driver.find_element(By.NAME,"rpaclprof_txtSrcMac").get_attribute("value") != "10:10:10:10:10:01"):
            print(f"             [Profile - ACL] FW Source MAC/OUI {driver.find_element(By.NAME,'rpaclprof_txtSrcMac').get_attribute('value')}!=10:10:10:10:10:01 Test Fail",file = open(file_name, '+a'))
            if_FW_error = True
            if_pro_new_error = True
        else:
            print(f"             [Profile - ACL] FW Source MAC/OUI 10:10:10:10:10:01 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.NAME,"rpaclprof_chkDrop").is_selected() == False):
            print(f"             [Profile - ACL] FW Drop {driver.find_element(By.NAME,'rpaclprof_chkDrop').is_selected()}!=True Test Fail",file = open(file_name, '+a'))
            if_pro_new_error = True
        else:
            print(f"             [Profile - ACL] FW Drop True Test Pass",file = open(file_name, '+a'))                
        # if(driver.find_element(By.NAME,"rpaclprof_txtUdfPriority").get_attribute("value") != '1'):
        #     print(f"             [Profile - ACL] FW Profile-Priority {driver.find_element(By.NAME,'rpaclprof_txtUdfPriority').get_attribute('value')}!=1 Test Fail",file = open(file_name, '+a'))
        #     if_pro_new_error = True
        # else:
        #     print(f"             [Profile - ACL] FW Profile-Priority 1 Test Pass",file = open(file_name, '+a'))               
        break
if not if_find:
    print(f"             [Profile - ACL] FW Not Found Profile Name 1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))    
    if_FW_error = True
# --------------------- FW web end ------------------------


# # Modify ACL Profile
# # --------------------- EMS web start ---------------------
# # Modify
# print(f"\n             [ACL - ACL Profile] EMS (Modify ACL Profile)",file = open(file_name, '+a'))
# driver.get(ems_url)
# time.sleep(1)
# mod.Login_EMS(driver,username,"Pqa1234!!")
# time.sleep(1)
# EMS_Provision_Config_path(driver,url)
# time.sleep(2)    
# driver.find_element_by_link_text("ACL").click()          
# time.sleep(1)
# driver.find_element_by_link_text("ACL Profile").click()
# time.sleep(3)
# print(f"             [ACL - ACL Profile] EMS Modify Profile-Priority False",file = open(file_name, '+a'))
# if_find = False
# for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
#     if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678001":
#         if_find = True
#         print(f"             [ACL - ACL Profile] EMS Profile Name 1234567890123456789012345678001 Found Test Pass",file = open(file_name, '+a'))
#         e_acl.ACL_Profile_Select(driver,str(i+1))
#         time.sleep(1)
#         e_acl.ACL_Profile_Modify(driver)
#         time.sleep(1)
#         e_mod.Checkbox_EMS(driver,"acl_aclprofile_pp_checkbox",False)
#         e_mod.OK(driver)
#         msg = alert_accept_text(driver)
#         if(if_msg_not_success(msg)):        
#             print(f"             [ACL - ACL Profile] EMS Modify Profile-Priority False  msg:{msg}  Test Fail",file = open(file_name, '+a'))
#             if_EMS_error = True
#             if_pro_modify_error = True
#         else:
#             print(f"             [ACL - ACL Profile] EMS Modify Profile-Priority False  msg:{msg}  Test Pass",file = open(file_name, '+a'))
#         break
# if not if_find:
#     print(f"             [ACL - ACL Profile] EMS Profile Name 1234567890123456789012345678001 Not Found Test Fail",file = open(file_name, '+a'))
#     print(f"             [ACL - ACL Profile] EMS Modify Profile-Priority False Test Fail",file = open(file_name, '+a'))
#     if_EMS_error = True   
#     if_pro_modify_error = True
# # Check
# print(f"             [ACL - ACL Profile] EMS check modified val",file = open(file_name, '+a'))
# time.sleep(1)
# EMS_Provision_Config_path(driver,url)
# time.sleep(2)    
# driver.find_element_by_link_text("ACL").click()          
# time.sleep(1)
# driver.find_element_by_link_text("ACL Profile").click()
# time.sleep(3)
# if_find = False
# for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
#     if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678001":
#         if_find = True
#         e_acl.ACL_Profile_Select(driver,i+1)
#         e_acl.ACL_Profile_Modify(driver)
#         time.sleep(1)
#         if(driver.find_element(By.XPATH,f'//*[@id="acl_aclprofile_pp_checkbox"]').is_selected() != False):
#             print(f'''             [ACL - ACL Profile] EMS Profile-Priority {driver.find_element(By.XPATH,f'//*[@id="acl_aclprofile_pp_checkbox"]').is_selected()}!=False Test Fail''',file = open(file_name, '+a'))
#             if_EMS_error = True
#             if_pro_modify_error = True
#         else:
#             print(f"             [ACL - ACL Profile] EMS Profile-Priority False Test Pass",file = open(file_name, '+a'))    
#         break
# if not if_find:
#     print(f"             [ACL - ACL Profile] EMS Not Found Profile Name 1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))    
#     if_EMS_error = True    
#     if_pro_modify_error = True
# # --------------------- EMS web end ---------------------
# # Check Modified ACL Profile
# # --------------------- FW web start ---------------------
# print(f"             [Profile - ACL] FW check modified val",file = open(file_name, '+a'))
# # Profile > ACL 
# driver.get(url) 
# time.sleep(3)
# mod.Login(driver,username,password)      
# mod.Switch_Default(driver)
# mod.Page_Switch(driver,"Profile")
# # mod.SubPage_Switch(driver,"ACL")
# mod.Switch_Frame(driver)
# time.sleep(2)    
# if_find = False
# for i in range(driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName").__len__()):
#     if driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName")[i].text == '1234567890123456789012345678001':
#         if_find = True     
#         driver.find_elements(By.NAME,"rpaclprof_rdoSlt")[i].click()
#         if(driver.find_element(By.NAME,"rpaclprof_chkUdfPriority").is_selected() != False):
#             print(f'             [Profile - ACL] FW Profile-Priority {driver.find_element(By.NAME,"rpaclprof_chkUdfPriority").is_selected()}!=False Test Fail',file = open(file_name, '+a'))
#             if_FW_error = True
#             if_pro_modify_error = True
#         else:
#             print(f"             [Profile - ACL] FW Profile-Priority False Test Pass",file = open(file_name, '+a'))               
#         break
# if not if_find:
#     print(f"             [Profile - ACL] FW Not Found Profile Name 1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))    
#     if_FW_error = True    
#     if_pro_modify_error = True
# # --------------------- FW web end ---------------------
# if if_pro_modify_error:
#     print(f"  [EMS1-6148][ACL - ACL Profile - Modify] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6148][ACL - ACL Profile - Modify] Result Pass",file = open(file_name, '+a'))  


# ACL Rule
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Rule] EMS (Set ACL Rule and Check)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(3)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port)

if "NXC" in cardtype:
    # (NXC NNI Port) + (ACL Profile with priority) Should Apply Fail Test :
    print(f"             [ACL - ACL Rule] EMS (NXC NNI Port) + (ACL Profile with priority) Should Apply Fail Test :",file = open(file_name, '+a'))
    e_acl.ACL_Rule_ACL_Profile(driver,"12345678901234567890_priority_1")
    e_acl.ACL_Rule_ACL_Priority(driver,"1")
    e_acl.ACL_Rule_Apply(driver)
    msg = alert_accept_text(driver)
    if(if_msg_not_success(msg)):
        print(f"             [ACL - ACL Rule]       EMS Slot-Port:{slot}-{port}, Profile:12345678901234567890_priority_1, Priority:1 Apply msg:{msg} Test Pass",file = open(file_name, '+a'))
    else:
        print(f"             [ACL - ACL Rule]       EMS Slot-Port:{slot}-{port}, Profile:12345678901234567890_priority_1, Priority:1 Apply msg:{msg} Test Fail",file = open(file_name, '+a'))
        if_EMS_error = True
        if_ru_priority_error = True


e_acl.ACL_Rule_ACL_Profile(driver,"1234567890123456789012345678001")
e_acl.ACL_Rule_ACL_Priority(driver,"1")
e_acl.ACL_Rule_Apply(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):
    print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:1234567890123456789012345678001, Priority:1 Apply msg:{msg} Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_ru_priority_error = True
else:
    print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:1234567890123456789012345678001, Priority:1 Apply msg:{msg} Test Pass",file = open(file_name, '+a'))
# Check
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(3)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port)    
time.sleep(3)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile")[i].text == "1234567890123456789012345678001":
        if_find = True
        print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789012345678001 Found Test Pass",file = open(file_name, '+a'))
        t = driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_priority")[i].text
        if(t != '1'): 
            print(f"             [ACL - ACL Rule] ACL Priority {t}!=1 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_ru_priority_error = True
        else:
            print(f"             [ACL - ACL Rule] ACL Priority 1 Test Pass",file = open(file_name, '+a'))
        break
if not if_find:
    print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789012345678001 Not Found Test Fail",file = open(file_name, '+a'))
    print(f"             [ACL - ACL Rule] EMS Priority 1 Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True          
    if_ru_priority_error = True
# --------------------- EMS web end ----------------------
# Check ACL Rule
# --------------------- FW web start ---------------------
# ACL > ACL Rule
driver.get(url)  
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"ACL")
mod.SubPage_Switch(driver,"ACL Rule")
mod.Switch_Frame(driver)
time.sleep(2)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selSlot']/option[{slot}]").click()
time.sleep(1)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selPort']/option[{port}]").click()
time.sleep(1)    
driver.find_element(By.NAME,"lctbl_selLoad").click()
time.sleep(5)    
t = driver.find_elements(By.NAME,"rpaclrule_portprofilename")[0].text
if t != "1234567890123456789012345678001":
    print(f"             [ACL - ACL Rule] FW ACL Profile Name {t}!=1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))
    print(f"             [ACL - ACL Rule] FW Priority 1 Test Fail",file = open(file_name, '+a'))
    if_FW_error = True    
    if_ru_priority_error = True
else:
    print(f"             [ACL - ACL Rule] FW ACL Profile Name 1234567890123456789012345678001 Test Pass",file = open(file_name, '+a'))
    print(f"             [ACL - ACL Rule] FW Priority 1 Test Pass",file = open(file_name, '+a'))
# --------------------- FW web end -----------------------
# if if_ru_profile_error:
#     print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Pass",file = open(file_name, '+a'))   
if if_ru_priority_error:
    print(f"  [EMS1-6137][ACL - ACL Rule - Apply with Priority] Result Fail",file = open(file_name, '+a'))
else:
    print(f"  [EMS1-6137][ACL - ACL Rule - Apply with Priority] Result Pass",file = open(file_name, '+a'))


# Delete ACL Rule
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Rule] EMS (Delete ACL Rule and Check)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(2)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port)
# e_acl.ACL_Rule_Select(driver,"1") 
time.sleep(1)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile")[i].text == "1234567890123456789012345678001":
        if_find = True
        e_acl.ACL_Rule_Select(driver,str(i+1)) 
        e_acl.ACL_Rule_Delete(driver) 
        msg = alert_accept_text(driver)
        if(if_msg_not_success(msg)):
            print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile: 1234567890123456789012345678001, Priority:1 Delete msg:{msg} Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_ru_delete_error = True
        else:
            print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile: 1234567890123456789012345678001, Priority:1 Delete msg:{msg} Test Pass",file = open(file_name, '+a'))
        break
if not if_find:
    print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789012345678001 Not Found Before Delete Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_ru_delete_error = True
# Check
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(2)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port)   
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile")[i].text == "1234567890123456789012345678001":
        if_find = True
        break
if if_find:
    print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789012345678001 Found After Delete Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_ru_delete_error = True
else:
    print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789012345678001 Not Found After Delete Test Pass",file = open(file_name, '+a'))
# --------------------- EMS web end ---------------------
# Check Delete ACL Rule
# --------------------- FW web start ---------------------
# ACL > ACL Rule
driver.get(url)  
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"ACL")
mod.SubPage_Switch(driver,"ACL Rule")
mod.Switch_Frame(driver)
time.sleep(2)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selSlot']/option[{slot}]").click()
time.sleep(1)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selPort']/option[{port}]").click()
time.sleep(1)    
driver.find_element(By.NAME,"lctbl_selLoad").click()
time.sleep(3)    
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclrule_portprofilename").__len__()):
    if driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclrule_portprofilename")[i].text == "1234567890123456789012345678001":
        if_find = True
        print(f"             [ACL - ACL Rule] FW ACL Profile Name 1234567890123456789012345678001 Found Delete Test Fail",file = open(file_name, '+a'))
        if_FW_error = True
        if_ru_delete_error = True
        break
if not if_find:
    print(f"             [ACL - ACL Rule] FW ACL Profile Name 1234567890123456789012345678001 Not Found Delete Test Pass",file = open(file_name, '+a'))
# --------------------- FW web end -----------------------
# if if_ru_delete_error:
#     print(f"  [EMS1-6138][ACL - ACL Rule - Delete] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6138][ACL - ACL Rule - Delete] Result Pass",file = open(file_name, '+a'))   


# Delete One ACL Profile
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Profile] EMS (Delete One ACL Profile and Check)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678001":
        if_find = True
        e_acl.ACL_Profile_Select(driver,i+1)
        e_acl.ACL_Profile_Delete(driver)
        msg = alert_accept_text(driver)
        if(if_msg_not_success(msg)):        
            print(f"             [ACL - ACL Profile] EMS Delete Profile 1234567890123456789012345678001 msg:{msg}  Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_delete_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Delete Profile  1234567890123456789012345678001 msg:{msg}  Test Pass",file = open(file_name, '+a'))
        break
if not if_find:
    print(f"             [ACL - ACL Profile] EMS Not Found Profile Name 1234567890123456789012345678001 Before Delete Test Fail",file = open(file_name, '+a'))    
    if_EMS_error = True
    if_pro_delete_error = True
# Check
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678001":
        if_find = True
        break
if if_find:
    print(f"             [ACL - ACL Profile] EMS Found Profile Name 1234567890123456789012345678001 After Delete Test Fail",file = open(file_name, '+a'))    
    if_EMS_error = True
    if_pro_delete_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Not Found Profile Name 1234567890123456789012345678001 After Delete Test Pass",file = open(file_name, '+a'))    
# --------------------- EMS web end ---------------------
# Check Delete One ACL Profile
# --------------------- FW web start ---------------------
# Profile > ACL 
driver.get(url) 
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"Profile")
# mod.SubPage_Switch(driver,"ACL")
mod.Switch_Frame(driver)
time.sleep(1)    
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName").__len__()):
    if driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName")[i].text == '1234567890123456789012345678001':
        if_find = True
        break
if if_find:
    print(f"             [Profile - ACL] FW Found Profile Name 1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))    
    if_FW_error = True
    if_pro_delete_error = True
else:
    print(f"             [Profile - ACL] FW Not Found Profile Name 1234567890123456789012345678001 Test Pass",file = open(file_name, '+a'))    
# --------------------- FW web end ------------------------    

# Delete All ACL Profile
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Profile] EMS (Delete All ACL Profile and Check)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
e_acl.ACL_Profile_Select(driver)
e_acl.ACL_Profile_Delete(driver)      
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS Delete All Profile msg:{msg}  Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_delete_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Delete All Profile msg:{msg}  Test Pass",file = open(file_name, '+a'))
time.sleep(1)
try:
    e_acl.ACL_Profile_Select(driver)
    e_acl.ACL_Profile_Delete(driver)
    msg = alert_accept_text(driver)
    # if(if_msg_not_success(msg)):        
    #     print(f"             [ACL - ACL Profile] EMS Second Time Delete All Profile msg:{msg}  Test Fail",file = open(file_name, '+a'))
    #     if_EMS_error = True
    #     if_pro_delete_error = True
    # else:
    #     print(f"             [ACL - ACL Profile] EMS Second Time Delete All Profile msg:{msg}  Test Pass",file = open(file_name, '+a'))
except:
    print(f"             [ACL - ACL Profile] EMS Second Time Select All Delete All Action Fail",file = open(file_name, '+a'))
# Check
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
table_row = driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()
if table_row != 0:
    print(f"             [ACL - ACL Profile] EMS Table Row Num {table_row}!=0 After Delete Test Fail",file = open(file_name, '+a'))    
    if_EMS_error = True
    if_pro_delete_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Table Row Num {table_row}=0 After Delete Test Pass",file = open(file_name, '+a'))    
# --------------------- EMS web start ---------------------
# Check Delete All ACL Profile
# --------------------- FW web start ---------------------
# Profile > ACL 
driver.get(url) 
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"Profile")
# mod.SubPage_Switch(driver,"ACL")
mod.Switch_Frame(driver)
time.sleep(1)    
table_row = driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName").__len__()
if table_row != 0:
    print(f"             [Profile - ACL] FW Table Row Num {table_row}!=0 After Delete Test Fail",file = open(file_name, '+a'))    
    if_FW_error = True
    if_pro_delete_error = True
else:
    print(f"             [Profile - ACL] FW Table Row Num {table_row}=0 After Delete Test Pass",file = open(file_name, '+a'))    
# --------------------- FW web end ------------------------ 
# if if_pro_delete_error:
#     print(f"  [EMS1-6150][ACL - ACL Profile - Delete] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6150][ACL - ACL Profile - Delete] Result Pass",file = open(file_name, '+a'))  
    
            
# if(if_EMS_error == True):
#     print(f"             [ACL - ACL Profile] EMS web check fail",file = open(file_name, '+a'))
# else:
#     print(f"             [ACL - ACL Profile] EMS web check pass",file = open(file_name, '+a')) 
# if(if_FW_error == True):
#     print(f"             [Profile - ACL] FW web check fail",file = open(file_name, '+a'))
# else:
#     print(f"             [Profile - ACL] FW web check pass",file = open(file_name, '+a')) 
# if if_pro_new_error:
#     print(f"  [EMS1-6143][ACL - ACL Profile - New] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6143][ACL - ACL Profile - New] Result Pass",file = open(file_name, '+a'))
# if if_pro_mode_error:
#     print(f"  [EMS1-6142][ACL - ACL Profile - Mode Apply] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6142][ACL - ACL Profile - Mode Apply] Result Pass",file = open(file_name, '+a'))
# if if_pro_modify_error:
#     print(f"  [EMS1-6148][ACL - ACL Profile - Modify] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6148][ACL - ACL Profile - Modify] Result Pass",file = open(file_name, '+a'))      
# if if_pro_delete_error:
#     print(f"  [EMS1-6150][ACL - ACL Profile - Delete] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6150][ACL - ACL Profile - Delete] Result Pass",file = open(file_name, '+a'))   
# if if_ru_profile_error:
#     print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Pass",file = open(file_name, '+a'))   
# if if_ru_priority_error:
#     print(f"  [EMS1-6137][ACL - ACL Rule - Apply with Priority] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6137][ACL - ACL Rule - Apply with Priority] Result Pass",file = open(file_name, '+a'))   
# if if_ru_delete_error:
#     print(f"  [EMS1-6138][ACL - ACL Rule - Delete] Result Fail",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-6138][ACL - ACL Rule - Delete] Result Pass",file = open(file_name, '+a'))                                                
# if((if_EMS_error == False) and (if_FW_error == False)):
#     print(f"  [EMS1-0000][ACL - ACL Rule] Result Pass",file = open(file_name, '+a'))
#     print(f"  [EMS1-0000][ACL - ACL Profile] Result Pass",file = open(file_name, '+a'))
# else:
#     print(f"  [EMS1-0000][ACL - ACL Rule] Result Fail",file = open(file_name, '+a'))
#     print(f"  [EMS1-0000][ACL - ACL Profile] Result Fail",file = open(file_name, '+a'))

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

# ACL Profile Mode : Profile
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Profile] EMS (Set ACL Profile Mode : Profile)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
print(f"             [ACL - ACL Profile] EMS set Mode:Profile",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Name:1234567890123456789012345678100",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Source MAC/OUI:10:10:10:10:10:01",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Drop:True",file = open(file_name, '+a'))
print(f"             [ACL - ACL Profile] EMS set Profile-Priority:1",file = open(file_name, '+a'))
e_acl.ACL_Profile_Mode(driver,"Profile")
e_acl.ACL_Profile_Mode_Apply(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Profile  Apply msg:{msg}  Apply Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_mode_error = True
else:
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Profile  Apply msg:{msg}  Apply Pass",file = open(file_name, '+a'))
e_acl.ACL_Profile_New(driver)
e_acl.ACL_Profile_New_Name(driver,"1234567890123456789012345678100")
e_acl.ACL_Profile_New_Source_MAC_OUI(driver,"10:10:10:10:10:01")
e_acl.ACL_Profile_New_Drop(driver)
e_acl.ACL_Profile_New_Profile_Priority(driver,"1")
e_mod.OK(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789012345678100 msg:{msg}  Apply Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_new_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789012345678100 msg:{msg}  Apply Pass",file = open(file_name, '+a'))
# Check
print(f"             [ACL - ACL Profile] (Check)",file = open(file_name, '+a'))
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Profile").click()
time.sleep(2)
if(Select(driver.find_element(By.ID,"acl_AclProfile_mode")).first_selected_option.text != "Profile"):
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : {Select(driver.find_element(By.ID,'acl_AclProfile_mode')).first_selected_option.text}!=Profile Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_mode_error = True
else:
    print(f"             [ACL - ACL Profile] EMS ACL Profile Mode : Profile Test Pass",file = open(file_name, '+a'))
time.sleep(1)
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_AclProfile_Name")[i].text == "1234567890123456789012345678100":
        if_find = True
        e_acl.ACL_Profile_Select(driver,i+1)
        e_acl.ACL_Profile_Modify(driver)
        time.sleep(1)
        if(driver.find_element(By.ID,"acl_aclprofile_name_text").get_attribute("value") != "1234567890123456789012345678100"):
            print(f"             [ACL - ACL Profile] EMS Name {driver.find_element(By.ID,'acl_aclprofile_name_text').get_attribute('value')}!=1234567890123456789012345678001 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Name 1234567890123456789012345678100 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.ID,"acl_aclprofile_sourmac_text").get_attribute("value") != "10:10:10:10:10:01"):
            print(f"             [ACL - ACL Profile] EMS Source MAC/OUI {driver.find_element(By.ID,'acl_aclprofile_sourmac_text').get_attribute('value')}!=10:10:10:10:10:01 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Source MAC/OUI 10:10:10:10:10:01 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.ID,"acl_aclprofile_drop_checkbox").is_selected() == False):
            print(f"             [ACL - ACL Profile] EMS Drop {driver.find_element(By.ID,'acl_aclprofile_drop_checkbox').is_selected()}!=True Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Drop True Test Pass",file = open(file_name, '+a'))                
        if(driver.find_element(By.ID,"acl_aclprofile_pp_text").get_attribute("value") != '1'):
            print(f"             [ACL - ACL Profile] EMS Profile-Priority {driver.find_element(By.ID,'acl_aclprofile_pp_text').get_attribute('value')}!=1 Test Fail",file = open(file_name, '+a'))
            if_EMS_error = True
            if_pro_new_error = True
        else:
            print(f"             [ACL - ACL Profile] EMS Profile-Priority 1 Test Pass",file = open(file_name, '+a'))    
        break
if not if_find:
    print(f"             [ACL - ACL Profile] EMS Not Found Profile Name 1234567890123456789012345678100 Test Fail",file = open(file_name, '+a'))    
    if_EMS_error = True
e_mod.Close2(driver)

# ACL Profil no priority for NXC ACL Rule Test
e_acl.ACL_Profile_New(driver)
e_acl.ACL_Profile_New_Name(driver,"1234567890123456789_no_priority")
e_acl.ACL_Profile_New_Source_MAC_OUI(driver,"10:10:10:10:10:01")
e_acl.ACL_Profile_New_Drop(driver)
e_mod.OK(driver)
msg = alert_accept_text(driver)
if(if_msg_not_success(msg)):        
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789_no_priority msg:{msg}  Apply Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_pro_new_error = True
else:
    print(f"             [ACL - ACL Profile] EMS Apply Profile 1234567890123456789_no_priority msg:{msg}  Apply Pass",file = open(file_name, '+a'))

# --------------------- EMS web end ---------------------
# Check ACL Profile
# --------------------- FW web start ---------------------
# Profile > ACL 
driver.get(url) 
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"Profile")
# mod.SubPage_Switch(driver,"ACL")
mod.Switch_Frame(driver)
time.sleep(2)    
if(Select(driver.find_element(By.NAME,"rpaclprof_aclProfileMode")).first_selected_option.text != "Profile"):
    print(f"             [Profile - ACL] FW ACL Profile Mode : {Select(driver.find_element(By.NAME,'rpaclprof_aclProfileMode')).first_selected_option.text}!=Profile Test Fail",file = open(file_name, '+a'))
    if_FW_error = True
    if_pro_mode_error = True
else:
    print(f"             [Profile - ACL] FW ACL Profile Mode : Profile Test Pass",file = open(file_name, '+a'))
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName").__len__()):
    if driver.find_elements(By.CLASS_NAME,"td-table-content.rpaclprof_txtName")[i].text == '1234567890123456789012345678100':
        if_find = True
        driver.find_elements(By.NAME,"rpaclprof_rdoSlt")[i].click()
        if(driver.find_element(By.NAME,"rpaclprof_txtSrcMac").get_attribute("value") != "10:10:10:10:10:01"):
            print(f"             [Profile - ACL] FW Source MAC/OUI {driver.find_element(By.NAME,'rpaclprof_txtSrcMac').get_attribute('value')}!=10:10:10:10:10:01 Test Fail",file = open(file_name, '+a'))
            if_FW_error = True
            if_pro_new_error = True
        else:
            print(f"             [Profile - ACL] FW Source MAC/OUI 10:10:10:10:10:01 Test Pass",file = open(file_name, '+a'))
        if(driver.find_element(By.NAME,"rpaclprof_chkDrop").is_selected() == False):
            print(f"             [Profile - ACL] FW Drop {driver.find_element(By.NAME,'rpaclprof_chkDrop').is_selected()}!=True Test Fail",file = open(file_name, '+a'))
            if_FW_error = True
            if_pro_new_error = True
        else:
            print(f"             [Profile - ACL] FW Drop True Test Pass",file = open(file_name, '+a'))                
        if(driver.find_element(By.NAME,"rpaclprof_txtUdfPriority").get_attribute("value") != '1'):
            print(f"             [Profile - ACL] FW Profile-Priority {driver.find_element(By.NAME,'rpaclprof_txtUdfPriority').get_attribute('value')}!=1 Test Fail",file = open(file_name, '+a'))
            if_FW_error = True
            if_pro_new_error = True
        else:
            print(f"             [Profile - ACL] FW Profile-Priority 1 Test Pass",file = open(file_name, '+a'))               
        break
if not if_find:
    print(f"             [Profile - ACL] FW Not Found Profile Name 1234567890123456789012345678100 Test Fail",file = open(file_name, '+a'))    
    if_pro_new_error = True
    if_FW_error = True
# --------------------- FW web end ------------------------            
# ACL Rule Apply With ACL Profile Only
# --------------------- EMS web start ---------------------
print(f"\n             [ACL - ACL Rule] EMS (Set ACL Rule Apply With ACL Profile Only)",file = open(file_name, '+a'))
driver.get(ems_url)
time.sleep(1)
mod.Login_EMS(driver,username,ems_password)
time.sleep(1)
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(2)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port)
# e_acl.ACL_Rule_ACL_Profile(driver,"1234567890123456789012345678100")
e_acl.ACL_Rule_ACL_Profile(driver,"1234567890123456789_no_priority")
e_acl.ACL_Rule_Apply(driver)
msg = alert_accept_text(driver) 
if(if_msg_not_success(msg)):
    print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:1234567890123456789_no_priority  Apply msg:{msg} Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True
    if_ru_profile_error = True
else:
    print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:1234567890123456789012345678100 Apply msg:{msg} Test Pass",file = open(file_name, '+a'))
# Check
# print(f"             [ACL - ACL Rule] (Check)",file = open(file_name, '+a'))
EMS_Provision_Config_path(driver,url)
time.sleep(2)    
driver.find_element_by_link_text("ACL").click()          
time.sleep(1)
driver.find_element_by_link_text("ACL Rule").click()
time.sleep(2)
e_acl.ACL_Rule_Slot(driver,slot)
e_acl.ACL_Rule_Port(driver,port) 
time.sleep(3)
# (NXA and NXP not support priority)
# if "NXA" in cardtype or "GLC" in cardtype: 
# if "NXP" not in cardtype and "OLC" not in cardtype: 
if ("NXC" not in cardtype) and ("MSC" not in cardtype): 
    
    # Check Unable to Set Priority
    print(f"             [ACL - ACL Rule] EMS (Check Unable to Set Priority)",file = open(file_name, '+a'))
    t = driver.find_element(By.ID,"acl_aclrule_priority_combobox").get_attribute("disabled")   
    if(str(t) != "true"):
        print(f"             [ACL - ACL Rule] EMS Can Set Priority Test Fail",file = open(file_name, '+a'))
        if_EMS_error = True
        if_ru_profile_error = True
    else:
        print(f"             [ACL - ACL Rule] EMS Can Not Set Priority Test Pass",file = open(file_name, '+a'))
# e_acl.ACL_Rule_ACL_Profile(driver,"12345678901234567890_priority_1")
# e_acl.ACL_Rule_Apply(driver)
# msg = alert_accept_text(driver)
# if(if_msg_not_success(msg)):
#     print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:12345678901234567890_priority_1 Priority:1 Apply msg:{msg} Test Fail",file = open(file_name, '+a'))
#     if_EMS_error = True
#     if_ru_profile_error = True
# else:
#     print(f"             [ACL - ACL Rule] EMS Slot-Port:{slot}-{port}, Profile:12345678901234567890_priority_1 Priority:1 Apply msg:{msg} Test Pass",file = open(file_name, '+a'))
    

time.sleep(3)    
if_find = False
for i in range(driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile").__len__()):
    if driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_aclprofile")[i].text == "1234567890123456789_no_priority":
        if_find = True
        print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789_no_priority Found Test Pass",file = open(file_name, '+a'))
        t = driver.find_elements(By.CLASS_NAME,"zy_td.acl_aclprofile_table_priority")[i].text
        # if(t != '0'): 

        # if "NXP" not in cardtype and "OLC" not in cardtype: 
        if ("NXC" not in cardtype) and ("MSC" not in cardtype): 
            if(t != '-'): 
                print(f"             [ACL - ACL Rule] EMS ACL Priority {t}!=- Test Fail",file = open(file_name, '+a'))
                if_EMS_error = True
                if_ru_profile_error = True
            else:
                print(f"             [ACL - ACL Rule] EMS ACL Priority - Test Pass",file = open(file_name, '+a'))
        else: # NXC can set
            if(t != '1'): 
                print(f"             [ACL - ACL Rule] EMS ACL Priority {t}!=1 Test Fail",file = open(file_name, '+a'))
                if_EMS_error = True
                if_ru_profile_error = True
            else:
                print(f"             [ACL - ACL Rule] EMS ACL Priority 1 Test Pass",file = open(file_name, '+a'))
        break
if not if_find:
    print(f"             [ACL - ACL Rule] EMS Profile Name 1234567890123456789_no_priority Not Found Test Fail",file = open(file_name, '+a'))
    print(f"             [ACL - ACL Rule] EMS Priority - Test Fail",file = open(file_name, '+a'))
    if_EMS_error = True          
    if_ru_profile_error = True
# --------------------- EMS web end ----------------------
# Check ACL Rule
# --------------------- FW web start ---------------------
# ACL > ACL Rule
driver.get(url)  
time.sleep(3)
mod.Login(driver,username,password)      
mod.Switch_Default(driver)
mod.Page_Switch(driver,"ACL")
mod.SubPage_Switch(driver,"ACL Rule")
mod.Switch_Frame(driver)
time.sleep(3)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selSlot']/option[{slot}]").click()
time.sleep(1)    
driver.find_element(By.XPATH,f"//*[@name='lctbl_selPort']/option[{port}]").click()
time.sleep(1)    
driver.find_element(By.NAME,"lctbl_selLoad").click()
time.sleep(5)                  
# t = driver.find_elements(By.NAME,"rpaclrule_profileprofilename")[0].text
t = driver.find_elements(By.NAME,"rpaclrule_portprofilename")[0].text
if t != "1234567890123456789_no_priority":
    print(f"             [ACL - ACL Rule] FW ACL Profile Name {t}!=1234567890123456789_no_priority Test Fail",file = open(file_name, '+a'))
    if_FW_error = True    
    if_ru_profile_error = True
else:
    print(f"             [ACL - ACL Rule] FW ACL Profile Name 1234567890123456789_no_priority Test Pass",file = open(file_name, '+a'))
# --------------------- FW web end -----------------------
if if_ru_profile_error:
    print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Fail",file = open(file_name, '+a'))
else:
    print(f"  [EMS1-6136][ACL - ACL Rule - Apply with Profile] Result Pass",file = open(file_name, '+a'))   
